## Read Me

This is a flexible notebook for stepping through an annotation step by step given a specific scenario input. The notebook must be modified in order to select the directory and input file. This is not ideal, so we will have to come up with a better system!

## Set up

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
from pathlib import Path
import sys

In [2]:
ROOT_DIR = os.getcwd() + '/../'
sys.path.append(ROOT_DIR)

sys.path.append(ROOT_DIR+'src/')
print(ROOT_DIR)

/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../


In [71]:
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
import src.moral_projection as moral_projection

import src.core_process as core_process
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)
importlib.reload(prompts)
importlib.reload(node)
importlib.reload(utils)
importlib.reload(core_process)

<module 'src.core_process' from '/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../src/core_process.py'>

In [12]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Selet Scenario Input File

In [5]:
# set main paths
SCENARIO_DIR = ROOT_DIR + "scenarios_inputs/" + "cheung_variants/"
# DATA_DIR_HUMAN = ROOT_DIR + "human_data/" 
OUTPUT_DIR = ROOT_DIR + "annotated_outputs/" + "cheung_variants/"

In [6]:
#set scenario file filename
FILENAME = 'lifeboat2.json'

#select scenario and action choice
SCENARIO_ID = 1
ACT_ID = '1'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


It is 1841, and I am the captain of a ship that has just sunk after colliding with an iceberg.
Survivors managed to escape in two lifeboats. My lifeboat (Lifeboat A) contains me and several other
crew members and passengers. It is already at full capacity, and it cannot safely take on any
additional people. The other lifeboat (Lifeboat B) contains 61 passengers and crew members. Their
boat is dangerously overcrowded, and large waves during the rainstorm could cause it to sink. I am
the highest-ranking officer, so the people in Lifeboat B call out to me for instructions. After
assessing the situation, I conclude that the overcrowded lifeboat would likely become stable if
about 10 adults left the boat. However, the ocean water is extremely cold, and anyone forced into
the water would almost certainly freeze to death. My own lifeboat cannot take any additional
passengers. 




In [7]:
# print scenario json
print(json.dumps(scenario_json, indent=4))

{
    "id": 1,
    "scenario_title": "Lifeboat",
    "deontology_level": "1",
    "utility_level": "3",
    "text": "It is 1841, and I am the captain of a ship that has just sunk after colliding with an iceberg. Survivors managed to escape in two lifeboats. My lifeboat (Lifeboat A) contains me and several other crew members and passengers. It is already at full capacity, and it cannot safely take on any additional people. The other lifeboat (Lifeboat B) contains 61 passengers and crew members. Their boat is dangerously overcrowded, and large waves during the rainstorm could cause it to sink. I am the highest-ranking officer, so the people in Lifeboat B call out to me for instructions. After assessing the situation, I conclude that the overcrowded lifeboat would likely become stable if about 10 adults left the boat. However, the ocean water is extremely cold, and anyone forced into the water would almost certainly freeze to death. My own lifeboat cannot take any additional passengers.",

#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

In [13]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
this_act_I = this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
print(scenario_json['text'])
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)
print("\n\nScenario:")
print(this_scenario_Ziv)
print(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Ziv observed the order for 10 passengers from lifeboat B to be forced overboard.
order for 10 passengers from lifeboat B to be forced overboard
It is 1841, and I am the captain of a ship that has just sunk after colliding with an iceberg. Survivors managed to escape in two lifeboats. My lifeboat (Lifeboat A) contains me and several other crew members and passengers. It is already at full capacity, and it cannot safely take on any additional people. The other lifeboat (Lifeboat B) contains 61 passengers and crew members. Their boat is dangerously overcrowded, and large waves during the rainstorm could cause it to sink. I am the highest-ranking officer, so the people in Lifeboat B call out to me for instructions. After assessing the situation, I conclude that the overcrowded lifeboat would likely become stable if about 10 adults left the boat. However, the ocean water is extremely cold, and anyone forced into the water would almost certainly freeze to death. My own lifeb

In [93]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')
g.set_version('d6d780129095bb9540a7957befb2014ea42d92c1')



Graph g initialized and reset.


#### Step 0. Get entities

In [94]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.core_process.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]
g = returned_beings[3]
    
#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
Other crew members and passengers in Lifeboat A
10 passengers from Lifeboat B who would be forced overboard
51 remaining passengers and crew members in Lifeboat B


#### Step 1. Deontology / Action Value Scores

In [ ]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
deontic_value,g  = annotate_scenario.core_process.process_values_simple(this_scenario, this_act_I, this_act_I,g) 
print(deontic_value)

    

You are an expert judge of the moral virtues of different actions. Moral actions are those which humans consider virtuous, that consider others' wellbeing and happiness, and that are guided by principles of ethics. You will be given a sentence describing an action. You must rate thus action in terms of its moral virtue. Respond with a rating from -100 to 100 for this action, where -100 is the most morally wrong action you can possibly think of, and 100 is the most morally virtuous action you can possibly think of. 0 indicates a morally neutral action. Please return a json object with key:value pair of "score": rating. Please rate this action: order for 10 passengers from lifeboat B to be forced overboard
({'score': -95}, <src.node.Graph object at 0x1420d79a0>)


#### Step 2. Outcome Likelihoods

In [99]:
#Step 2. Outcomes
processed_events = annotate_scenario.core_process.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

10 passengers from Lifeboat B are forced overboard into the cold ocean water
The 10 passengers forced overboard die from hypothermia due to freezing water
Lifeboat B becomes less overcrowded and more stable
Survivors in Lifeboat B experience reduced risk of sinking
Survivors in Lifeboat B may feel fear, distress, or trauma from the forced removal of passengers
I face moral and ethical consequences for ordering passengers to be forced overboard
Tension and possible resentment arise among survivors towards me
Survivors in Lifeboat A remain at full capacity without additional passengers
Overall survival chances for remaining passengers in Lifeboat B improve


#### Step 3. Outcome Utilities

In [101]:
#Step 3. Outcome utilities

impacts_list = annotate_scenario.core_process.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: 10 passengers from Lifeboat B are forced overboard into the cold ocean water
Scored impacts for these beings:
['Ziv', 'Other crew members and passengers in Lifeboat A', '10 passengers from Lifeboat B who would be forced overboard', '51 remaining passengers and crew members in Lifeboat B']
Scored values:
[-50, 0, -100, 20]

Processing impacts of event: The 10 passengers forced overboard die from hypothermia due to freezing water
Scored impacts for these beings:
['Ziv', 'Other crew members and passengers in Lifeboat A', '10 passengers from Lifeboat B who would be forced overboard', '51 remaining passengers and crew members in Lifeboat B']
Scored values:
[-50, -40, -100, 20]

Processing impacts of event: Lifeboat B becomes less overcrowded and more stable
Scored impacts for these beings:
['Ziv', 'Other crew members and passengers in Lifeboat A', '10 passengers from Lifeboat B who would be forced overboard', '51 remaining passengers and crew members in Lifeboa

#### Step 4. Cause / Intend / Know Links

In [41]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: The man on the ladder is shot and dies.
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The man falls off the ladder.
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The 19 other passengers behind the man are able to climb onto the deck.
{'cause': 'yes', 'intend': 'yes', 'know': 'no'}
CKI links for I
C+I+K-

Processing event: I experience the psychological impact of shooting the man.
{'cause': 'yes', 'intend': 'no', 'know': 'no'}
CKI links for I
C+I-K-

Processing event: The other passengers witness the shooting.
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+

Processing event: The overall chance of survival for the group on the ferry increases.
{'cause': 'yes', 'intend': 'yes', 'know': 'no'}
CKI links for I
C+I+K-


#### Step 5. Write out the results

In [42]:
narrative = FILENAME.split('.')[0]
this_output_filename = f"{OUTPUT_DIR}/{narrative}_{SCENARIO_ID}_choice_{ACT_ID}.json"

In [43]:
#optional -- write out the results 
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename, g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: /Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../annotated_outputs/cheung_variants//rope_ladder_1_choice_1.json



/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../annotated_outputs/cheung_variants//rope_ladder_1_choice_1.json
